# Курсовая: классическое МО для химических дескрипторов

## Методология:
1. SI вычисляется на основе IC50 и CC50, поэтому при моделировании:
   - **для задач по SI`** нельзя использовать IC50 и CC50 как признаки;
   - **для задач по IC50`и CC50 ** нельзя использовать SI как признак.
2. Для регрессии целевых переменных с сильной асимметрией используется log1p.
3. Сравнение делается через **несколько моделей + кросс-валидацию + подбор гиперпараметров**.
4. Все результаты сохраняются в папку artifacts/, чтобы затем их было удобно включить в отчёт и репозиторий.


## План работы

1. Загрузить данные, убрать служебный индекс, проверить пропуски/дубликаты/константные признаки.  
2. Провести EDA по целевым переменным и дескрипторам, отдельно проверить асимметрию и выбросы.  
3. Зафиксировать **анти-утечку** признаков:
   - SI-задачи: без IC50, CC50;
   - IC50/CC50-задачи: без SI.
4. Построить бейзлайн и 3–4 сильные модели для каждой задачи.
5. Сравнить модели по единому протоколу:
   - регрессия: MAE, RMSE, R²;
   - классификация: ROC-AUC, F1, Balanced Accuracy, PR-AUC.
6. Выбрать лучшие решения, визуализировать ошибки и важность признаков.
7. Сохранить таблицы результатов и на их основе собрать финальный отчёт.


In [ ]:
# !pip install openpyxl
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    ExtraTreesRegressor,
    ExtraTreesClassifier,
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
)
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


In [ ]:
DATA_PATH = Path("data") / "Данные_для_курсовой_Классическое_МО.xlsx"
OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_JOBS = -1

df = pd.read_excel(DATA_PATH)
df.head()


In [ ]:
# Базовая очистка
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("shape:", df.shape)
print("duplicates:", df.duplicated().sum())
print("missing cells:", int(df.isna().sum().sum()))
print("columns with NA:", int((df.isna().sum() > 0).sum()))


In [ ]:
TARGETS = ["IC50, mM", "CC50, mM", "SI"]
TARGET_LIKE_COLUMNS = [
    "IC50, mM",
    "CC50, mM",
    "SI",
    "IC50_log",
    "CC50_log",
    "SI_log",
]
SI_TARGET_LIKE_NAMES = {
    "si",
    "si_log",
    "log_si",
    "si_bin",
    "si_class",
    "si_scaled",
    "target_si",
}
FEATURES_ALL = [c for c in df.columns if c not in TARGET_LIKE_COLUMNS]

target_summary = df[TARGETS].describe().T
target_summary["skew"] = df[TARGETS].skew()
target_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, TARGETS):
    ax.hist(df[col].dropna(), bins=40)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# Added log transformation
df["IC50_log"] = np.log10(df["IC50, mM"] + 1e-8)


In [ ]:
# Лог-шкала для наглядности
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, TARGETS):
    ax.hist(np.log1p(df[col].dropna()), bins=40)
    ax.set_title(f"log1p({col})")
plt.tight_layout()
plt.show()


## Правила формирования признаков без утечки

- Для задач **по IC50** исключаем: IC50, mM, SI
- Для задач **по CC50** исключаем: CC50, mM, SI
- Для задач **по SI** исключаем: SI, IC50, mM, CC50, mM


In [ ]:
def get_target_like_columns(columns):
    target_like = []
    explicit_targets = set(TARGET_LIKE_COLUMNS)
    for col in columns:
        normalized = col.lower()
        if (
            col in explicit_targets
            or "ic50" in normalized
            or "cc50" in normalized
            or normalized in SI_TARGET_LIKE_NAMES
        ):
            target_like.append(col)
    return target_like


def get_feature_columns(task_name: str):
    if not (task_name.startswith("reg_") or task_name.startswith("cls_")):
        raise ValueError(f"Unknown task: {task_name}")
    target_like = set(get_target_like_columns(df.columns))
    return [c for c in df.columns if c not in target_like]


def validate_no_target_like_columns(X):
    leaked = get_target_like_columns(X.columns)
    if leaked:
        raise ValueError(f"Target leakage detected in features: {leaked}")


def assert_no_target_like_columns(X):
    leaked = get_target_like_columns(X.columns)
    assert not leaked, f"Feature matrix contains target-like columns: {leaked}"


In [ ]:
# Медиа́ны для классификационных задач
med_ic50 = df["IC50, mM"].median()
med_cc50 = df["CC50, mM"].median()
med_si = df["SI"].median()

classification_targets = {
    "cls_IC50_gt_median": (df["IC50, mM"] > med_ic50).astype(int),
    "cls_CC50_gt_median": (df["CC50, mM"] > med_cc50).astype(int),
    "cls_SI_gt_median": (df["SI"] > med_si).astype(int),
    "cls_SI_gt_8": (df["SI"] > 8).astype(int),
}

{
    "IC50_median": med_ic50,
    "CC50_median": med_cc50,
    "SI_median": med_si,
    "SI_threshold": 8,
}


In [ ]:
# EDA по классам
for name, y in classification_targets.items():
    print(name, y.value_counts(normalize=True).rename("share").round(3).to_dict())


In [ ]:
def make_regression_models():
    return {
        "elastic_net": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", TransformedTargetRegressor(
                regressor=ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
                func=np.log1p,
                inverse_func=np.expm1,
            )),
        ]),
        "random_forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", TransformedTargetRegressor(
                regressor=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
                func=np.log1p,
                inverse_func=np.expm1,
            )),
        ]),
        "extra_trees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", TransformedTargetRegressor(
                regressor=ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
                func=np.log1p,
                inverse_func=np.expm1,
            )),
        ]),
        "hist_gbr": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", TransformedTargetRegressor(
                regressor=HistGradientBoostingRegressor(random_state=RANDOM_STATE),
                func=np.log1p,
                inverse_func=np.expm1,
            )),
        ]),
    }

def make_classification_models():
    return {
        "log_reg": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=RANDOM_STATE,
                max_iter=5000,
                class_weight="balanced",
            )),
        ]),
        "random_forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
                class_weight="balanced",
            )),
        ]),
        "extra_trees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
                class_weight="balanced",
            )),
        ]),
        "hist_gbc": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                random_state=RANDOM_STATE,
            )),
        ]),
    }


In [ ]:
REG_PARAM_DISTS = {
    "elastic_net": {
        "model__regressor__alpha": np.logspace(-3, 1, 20),
        "model__regressor__l1_ratio": np.linspace(0.05, 0.95, 10),
    },
    "random_forest": {
        "model__regressor__n_estimators": [300, 500, 800],
        "model__regressor__max_depth": [None, 6, 10, 16, 24],
        "model__regressor__min_samples_leaf": [1, 2, 5, 10],
        "model__regressor__max_features": ["sqrt", "log2", 0.5, 0.8],
    },
    "extra_trees": {
        "model__regressor__n_estimators": [300, 500, 800],
        "model__regressor__max_depth": [None, 6, 10, 16, 24],
        "model__regressor__min_samples_leaf": [1, 2, 5, 10],
        "model__regressor__max_features": ["sqrt", "log2", 0.5, 0.8],
    },
    "hist_gbr": {
        "model__regressor__learning_rate": np.linspace(0.02, 0.2, 10),
        "model__regressor__max_depth": [None, 3, 5, 7, 10],
        "model__regressor__max_leaf_nodes": [15, 31, 63, 127],
        "model__regressor__min_samples_leaf": [10, 20, 30, 50],
        "model__regressor__l2_regularization": np.logspace(-6, 0, 10),
    },
}

CLS_PARAM_DISTS = {
    "log_reg": {
        "model__C": np.logspace(-3, 2, 20),
    },
    "random_forest": {
        "model__n_estimators": [300, 500, 800],
        "model__max_depth": [None, 6, 10, 16, 24],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__max_features": ["sqrt", "log2", 0.5, 0.8],
    },
    "extra_trees": {
        "model__n_estimators": [300, 500, 800],
        "model__max_depth": [None, 6, 10, 16, 24],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__max_features": ["sqrt", "log2", 0.5, 0.8],
    },
    "hist_gbc": {
        "model__learning_rate": np.linspace(0.02, 0.2, 10),
        "model__max_depth": [None, 3, 5, 7, 10],
        "model__max_leaf_nodes": [15, 31, 63, 127],
        "model__min_samples_leaf": [10, 20, 30, 50],
        "model__l2_regularization": np.logspace(-6, 0, 10),
    },
}


In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred),
    }

def classification_metrics(y_true, y_pred, y_prob):
    return {
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_prob),
    }


In [ ]:
def fit_best_regression_model(X, y, model_name, model, n_iter=20, cv_splits=5):
    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=REG_PARAM_DISTS[model_name],
        n_iter=n_iter,
        scoring="neg_root_mean_squared_error",
        n_jobs=N_JOBS,
        cv=cv,
        random_state=RANDOM_STATE,
        refit=True,
    )
    validate_no_target_like_columns(X)
    assert_no_target_like_columns(X)
    search.fit(X, y)
    return search

def fit_best_classification_model(X, y, model_name, model, n_iter=20, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=CLS_PARAM_DISTS[model_name],
        n_iter=n_iter,
        scoring="roc_auc",
        n_jobs=N_JOBS,
        cv=cv,
        random_state=RANDOM_STATE,
        refit=True,
    )
    validate_no_target_like_columns(X)
    assert_no_target_like_columns(X)
    search.fit(X, y)
    return search


In [ ]:
def run_regression_task(task_name, target_col):
    feature_cols = get_feature_columns(task_name)
    X = df[feature_cols].copy()
    validate_no_target_like_columns(X)
    assert_no_target_like_columns(X)
    y = df[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    models = make_regression_models()
    rows = []
    fitted = {}

    for model_name, model in models.items():
        search = fit_best_regression_model(X_train, y_train, model_name, model)
        best_model = search.best_estimator_

        pred_train = best_model.predict(X_train)
        pred_test = best_model.predict(X_test)

        row = {
            "task": task_name,
            "model": model_name,
            "cv_best_score_rmse": -search.best_score_,
            "train_MAE": regression_metrics(y_train, pred_train)["MAE"],
            "train_RMSE": regression_metrics(y_train, pred_train)["RMSE"],
            "train_R2": regression_metrics(y_train, pred_train)["R2"],
            "test_MAE": regression_metrics(y_test, pred_test)["MAE"],
            "test_RMSE": regression_metrics(y_test, pred_test)["RMSE"],
            "test_R2": regression_metrics(y_test, pred_test)["R2"],
            "best_params": str(search.best_params_),
            "n_features": len(feature_cols),
        }
        rows.append(row)
        fitted[model_name] = {
            "search": search,
            "best_model": best_model,
            "X_test": X_test,
            "y_test": y_test,
            "pred_test": pred_test,
            "feature_cols": feature_cols,
        }

    results = pd.DataFrame(rows).sort_values(["test_RMSE", "test_MAE", "test_R2"], ascending=[True, True, False])
    results.to_csv(OUT_DIR / f"{task_name}_results.csv", index=False)
    return results, fitted


In [ ]:
def run_classification_task(task_name, y):
    feature_cols = get_feature_columns(task_name)
    X = df[feature_cols].copy()
    validate_no_target_like_columns(X)
    assert_no_target_like_columns(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    models = make_classification_models()
    rows = []
    fitted = {}

    for model_name, model in models.items():
        search = fit_best_classification_model(X_train, y_train, model_name, model)
        best_model = search.best_estimator_

        prob_train = best_model.predict_proba(X_train)[:, 1] if hasattr(best_model, "predict_proba") else best_model.decision_function(X_train)
        prob_test = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else best_model.decision_function(X_test)

        pred_train = (prob_train >= 0.5).astype(int)
        pred_test = (prob_test >= 0.5).astype(int)

        train_scores = classification_metrics(y_train, pred_train, prob_train)
        test_scores = classification_metrics(y_test, pred_test, prob_test)

        row = {
            "task": task_name,
            "model": model_name,
            "cv_best_roc_auc": search.best_score_,
            "train_ROC_AUC": train_scores["ROC_AUC"],
            "train_F1": train_scores["F1"],
            "train_Balanced_Accuracy": train_scores["Balanced_Accuracy"],
            "train_PR_AUC": train_scores["PR_AUC"],
            "test_ROC_AUC": test_scores["ROC_AUC"],
            "test_F1": test_scores["F1"],
            "test_Balanced_Accuracy": test_scores["Balanced_Accuracy"],
            "test_PR_AUC": test_scores["PR_AUC"],
            "best_params": str(search.best_params_),
            "n_features": len(feature_cols),
        }
        rows.append(row)
        fitted[model_name] = {
            "search": search,
            "best_model": best_model,
            "X_test": X_test,
            "y_test": y_test,
            "prob_test": prob_test,
            "pred_test": pred_test,
            "feature_cols": feature_cols,
        }

    results = pd.DataFrame(rows).sort_values(["test_ROC_AUC", "test_PR_AUC", "test_F1"], ascending=[False, False, False])
    results.to_csv(OUT_DIR / f"{task_name}_results.csv", index=False)
    return results, fitted


## Запуск регрессионных задач

Ниже можно запускать задачи по одной, чтобы не перегружать ноутбук. Для финальной версии курсовой лучше сохранить результаты каждой задачи в отдельный `.py`-файл, но как исследовательский центр ноутбук полезен для перебора и сравнения.


In [ ]:
# Запуск: регрессия IC50
reg_ic50_results, reg_ic50_fitted = run_regression_task("reg_IC50", "IC50, mM")
reg_ic50_results


In [ ]:
# Запуск: регрессия CC50
reg_cc50_results, reg_cc50_fitted = run_regression_task("reg_CC50", "CC50, mM")
reg_cc50_results


In [ ]:
# Запуск: регрессия SI
reg_si_results, reg_si_fitted = run_regression_task("reg_SI", "SI")
reg_si_results


## Запуск классификационных задач

In [ ]:
# IC50 > median
cls_ic50_results, cls_ic50_fitted = run_classification_task(
    "cls_IC50_gt_median",
    classification_targets["cls_IC50_gt_median"]
)
cls_ic50_results


In [ ]:
# CC50 median
cls_cc50_results, cls_cc50_fitted = run_classification_task(
     "cls_CC50_gt_median",
     classification_targets["cls_CC50_gt_median"]
 )
cls_cc50_results

In [ ]:
# SI median
cls_si_median_results, cls_si_median_fitted = run_classification_task(
     "cls_SI_gt_median",
     classification_targets["cls_SI_gt_median"]
 )
cls_si_median_results

In [ ]:
# SI 8
cls_si_8_results, cls_si_8_fitted = run_classification_task(
     "cls_SI_gt_8",
     classification_targets["cls_SI_gt_8"]
 )
cls_si_8_results

In [ ]:
def plot_regression_diagnostics(fitted_bundle, model_name):
    data = fitted_bundle[model_name]
    y_true = data["y_test"]
    X_test = data["X_test"]
    model = data["best_model"]

    y_pred = model.predict(X_test)

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.6)
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.title(f"Regression diagnostics: {model_name}")

    line_min = min(y_true.min(), y_pred.min())
    line_max = max(y_true.max(), y_pred.max())
    plt.plot([line_min, line_max], [line_min, line_max])

    plt.tight_layout()
    plt.show()
# plot_regression_diagnostics(reg_ic50_fitted, "extra_trees")


In [ ]:
plot_regression_diagnostics(reg_ic50_fitted, "extra_trees")
plt.show()

In [ ]:
def show_top_importances(fitted_bundle, model_name, n_top=20):
    data = fitted_bundle[model_name]
    model = data["best_model"]
    X_test = data["X_test"]
    y_test = data["y_test"]
    feature_cols = data["feature_cols"]

    result = permutation_importance(
        model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=N_JOBS
    )

    imp = (
        pd.DataFrame({
            "feature": feature_cols,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
        })
        .sort_values("importance_mean", ascending=False)
        .head(n_top)
    )

    plt.figure(figsize=(8, 6))
    plt.barh(imp["feature"][::-1], imp["importance_mean"][::-1])
    plt.title(f"Top-{n_top} importances: {model_name}")
    plt.tight_layout()
    plt.show()

    return imp

# show_top_importances(reg_ic50_fitted, "extra_trees", n_top=20)


## В финальный отчёт выносим:

1. **Описание данных**: размер выборки, число признаков, пропуски, асимметрия таргетов.  
2. **Анти-утечка**: отдельно объяснить, почему SI нельзя использовать как признак в задачах по IC50/CC50, а IC50 и CC50 — в задачах по SI.  
3. **Таблица сравнения моделей** по каждой из 7 задач.  
4. **Ключевые графики**:
   - распределения таргетов до/после log1p;
   - факт vs прогноз для регрессии;
   - важность признаков;
   - при желании ROC/PR-кривые для классификации.
5. **Итоговый вывод**:
   - какая модель лучшая для каждой задачи;
   - где есть переобучение;
   - какие признаки наиболее влиятельны;
   - что улучшать дальше (feature selection, outlier treatment, target transformation, stacking/boosting).


## Структура проекта для GitHub

```text
├── README.md
├── data/
│   └── raw/
├── notebooks/
│   └── course_project_workflow.ipynb
├── src/
│   ├── eda.py
│   ├── reg_ic50.py
│   ├── reg_cc50.py
│   ├── reg_si.py
│   ├── cls_ic50_median.py
│   ├── cls_cc50_median.py
│   ├── cls_si_median.py
│   └── cls_si_gt8.py
├── artifacts/
│   ├── *.csv
│   └── *.png
└── report/
    └── report.pdf
```
